In [0]:
dbutils.widgets.text("catalog", "")
catalog = dbutils.widgets.get("catalog").strip()

In [0]:
# dbutils.widgets.text("catalog", "")
# catalog = dbutils.widgets.get("catalog").strip()
assert catalog, "catalog parameter required"
spark.sql(f"USE CATALOG `{catalog}`")

# Create the table once (idempotent)
spark.sql("""
CREATE TABLE IF NOT EXISTS ops.reconciliation_results (
  run_ts       TIMESTAMP,
  check_name   STRING,
  source_value DECIMAL(18,2),
  target_value DECIMAL(18,2),
  status       STRING
) COMMENT 'Ops: automated reconciliation of Bronze, Silver and Gold, five checks appended on every pipeline run'
""")

# Append five rows, one per check
spark.sql("""
INSERT INTO ops.reconciliation_results
WITH checks AS (

  SELECT 'ORDER_COUNT' AS check_name,
    (SELECT CAST(COUNT(DISTINCT nullif(upper(trim(order_id)), '')) AS DECIMAL(18,2))
       FROM bronze.orders) AS source_value,
    (SELECT CAST(COUNT(*) AS DECIMAL(18,2)) FROM silver.orders)
  + (SELECT CAST(COUNT(*) AS DECIMAL(18,2)) FROM silver.quarantine_orders) AS target_value

  UNION ALL
  SELECT 'LINE_COUNT',
    (SELECT CAST(COUNT(*) AS DECIMAL(18,2)) FROM silver.order_items),
    (SELECT CAST(COUNT(*) AS DECIMAL(18,2)) FROM gold.fact_orders)

  UNION ALL
  SELECT 'NET_REVENUE',
    (SELECT CAST(COALESCE(SUM(i.net_amount), 0) AS DECIMAL(18,2))
       FROM silver.order_items i
       JOIN silver.orders o ON i.order_id = o.order_id
      WHERE o.order_status = 'DELIVERED'),
    (SELECT CAST(COALESCE(SUM(net_item_revenue), 0) AS DECIMAL(18,2))
       FROM gold.agg_daily_store_sales)

  UNION ALL
  SELECT 'DELIVERED_ORDERS',
    (SELECT CAST(COUNT(*) AS DECIMAL(18,2)) FROM gold.fact_orders WHERE order_status = 'DELIVERED'),
    (SELECT CAST(COALESCE(SUM(delivered_orders), 0) AS DECIMAL(18,2)) FROM gold.agg_daily_store_sales)

  UNION ALL
  SELECT 'CURRENT_CUSTOMERS',
    (SELECT CAST(COUNT(DISTINCT customer_id) AS DECIMAL(18,2)) FROM silver.customers_type2),
    (SELECT CAST(COUNT(*) AS DECIMAL(18,2)) FROM gold.dim_customer WHERE is_current)
)
SELECT current_timestamp() AS run_ts,
       check_name,
       source_value,
       target_value,
       CASE WHEN source_value = target_value THEN 'PASS' ELSE 'FAIL' END AS status
FROM checks
""")

display(spark.sql("""
  SELECT * FROM ops.reconciliation_results
  WHERE run_ts = (SELECT MAX(run_ts) FROM ops.reconciliation_results)
  ORDER BY check_name"""))

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("checkpoint_name", "ADHOC")
dbutils.widgets.text("job_run_id", "MANUAL")

catalog         = dbutils.widgets.get("catalog").strip()
checkpoint_name = dbutils.widgets.get("checkpoint_name").strip()
assert catalog, "catalog parameter required"

In [0]:

spark.sql(f"USE CATALOG `{catalog}`")

spark.sql("""
CREATE TABLE IF NOT EXISTS ops.checkpoint_counts (
  checkpoint_name STRING,
  table_name      STRING,
  row_count       BIGINT,
  captured_at     TIMESTAMP
) COMMENT 'Ops: row counts of key tables captured after each pipeline run, used to prove incremental and idempotent loads'
""")

H2_TABLES = [
  "bronze.orders", "bronze.customers", "bronze.products", "bronze.stores",
  "silver.orders", "silver.order_items", "silver.quarantine_orders",
  "silver.customers_type2", "silver.products_type1",
  "gold.dim_customer", "gold.fact_orders",
  "gold.agg_daily_store_sales",
]

counts_sql = " UNION ALL ".join(
    f"SELECT '{t}' AS table_name, COUNT(*) AS row_count FROM {t}" for t in H2_TABLES)

spark.sql(f"""
INSERT INTO ops.checkpoint_counts
SELECT '{checkpoint_name}', table_name, row_count, current_timestamp()
FROM ({counts_sql})
""")

In [0]:
dbutils.widgets.text("job_run_id", "MANUAL")
job_run_id = dbutils.widgets.get("job_run_id").strip() or "MANUAL"

In [0]:
from datetime import datetime


spark.sql(f"USE CATALOG `{catalog}`")

spark.sql("""
CREATE TABLE IF NOT EXISTS ops.load_audit (
  job_run_id   STRING,
  task_name    STRING,
  target_table STRING,
  row_count    BIGINT,
  completed_at TIMESTAMP
) COMMENT 'Ops: one row per table written by each task on every job run'
""")

def log_audit(task_name, tables):
    """Append one audit row per table this task wrote."""
    rows = [(job_run_id, task_name, t, spark.table(t).count(), datetime.now())
            for t in tables]
    spark.createDataFrame(
        rows,
        "job_run_id string, task_name string, target_table string, row_count bigint, completed_at timestamp"
    ).write.mode("append").saveAsTable("ops.load_audit")

In [0]:
# ingest_bronze
log_audit("ingest_bronze", ["bronze.stores", "bronze.customers", "bronze.products", "bronze.orders"])

# silver_reference
log_audit("silver_reference", ["silver.stores", "silver.products_type1", "silver.customers_type2"])

# silver_orders
log_audit("silver_orders", ["silver.orders", "silver.order_items", "silver.quarantine_orders"])

# gold_build
log_audit("gold_build", ["gold.dim_date", "gold.dim_store", "gold.dim_product", "gold.dim_customer",
                         "gold.fact_orders", "gold.agg_daily_store_sales"])

# quality_checks (call after the reconciliation and H2 inserts)
log_audit("quality_checks", ["ops.reconciliation_results", "ops.checkpoint_counts"])